In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"

import torch
torch.set_num_threads(6)
torch.set_num_interop_threads(1)

In [2]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [3]:
set_seed(seed=777)

In [4]:
excel_file = pd.ExcelFile("../../../data/bpic19.xlsx", engine="openpyxl")

df = pd.concat(
    [
        pd.read_excel(
            excel_file,
            keep_default_na=False,
            dtype={
                "case:concept:name": "string",
                "concept:name": "string",
                "case:Spend area text": "string",
                "case:Document Type": "string",
                "case:Sub spend area text": "string",
                "case:Purch. Doc. Category name": "string",
                "case:Item Type": "string",
                "case:Item Category": "string",
                "case:Spend classification text": "string",
                "case:Source": "string",
                "case:GR-Based Inv. Verif.": "string",
                "case:Goods Receipt": "string",
                "Cumulative net worth (EUR)": "float32",
                "time_delta": "float32",
            }
        )
        for sheet in excel_file.sheet_names
    ],
    ignore_index=True,
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,Cumulative net worth (EUR),case:Document Type,case:GR-Based Inv. Verif.,case:Goods Receipt,case:Item Category,case:Item Type,case:Purch. Doc. Category name,case:Source,case:Spend area text,case:Spend classification text,case:Sub spend area text,concept:name,time_delta
0,2000000000_00001,2018-01-02 12:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Created,0.0
1,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Complete,3600.0
2,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Awaiting Approval,0.0
3,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Document Completed,0.0
4,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: In Transfer to Execution Syst.,0.0
5,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Ordered,0.0
6,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,SRM: Change was Transmitted,0.0
7,2000000000_00001,2018-01-02 13:53:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Create Purchase Order Item,0.0
8,2000000000_00001,2018-01-02 22:59:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Vendor creates invoice,32760.0
9,2000000000_00001,2018-03-06 06:44:00,298.0,EC Purchase order,False,True,"3-way match, invoice before GR",Standard,Purchase order,sourceSystemID_0000,CAPEX & SOCS,NPR,Facility Management,Record Goods Receipt,5384700.0


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [8]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['Cumulative net worth (EUR)', 'case:Document Type', 'case:GR-Based Inv. Verif.', 'case:Goods Receipt', 'case:Item Category', 'case:Item Type', 'case:Purch. Doc. Category name', 'case:Source', 'case:Spend area text', 'case:Spend classification text', 'case:Sub spend area text', 'concept:name', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
case:Document Type             categorical    case     yes    ['EC Purchase order', 'Framework order', 'Standard PO'] N/A        data_derived        
case:GR-Based Inv. Verif.      categorical    case     yes    ['False', 'True

In [9]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [10]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [11]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [12]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [13]:
engine.parallel_sets

[{'Change Delivery Indicator', 'Change Quantity'},
 {'Cancel Invoice Receipt',
  'Cancel Subsequent Invoice',
  'Clear Invoice',
  'Record Invoice Receipt',
  'Record Subsequent Invoice',
  'Remove Payment Block'},
 {'Cancel Invoice Receipt', 'Cancel Subsequent Invoice'},
 {'Change Currency', 'Change Price', 'Change payment term'},
 {'SRM: Change was Transmitted', 'SRM: Ordered'}]

In [14]:
engine.branching_sets

[{'Block Purchase Order Item',
  'Cancel Goods Receipt',
  'Cancel Invoice Receipt',
  'Cancel Subsequent Invoice',
  'Change Approval for Purchase Order',
  'Change Currency',
  'Change Delivery Indicator',
  'Change Price',
  'Change Quantity',
  'Change Rejection Indicator',
  'Change Storage Location',
  'Change payment term',
  'Clear Invoice',
  'Create Purchase Order Item',
  'Create Purchase Requisition Item',
  'Delete Purchase Order Item',
  'Reactivate Purchase Order Item',
  'Receive Order Confirmation',
  'Record Goods Receipt',
  'Record Invoice Receipt',
  'Record Service Entry Sheet',
  'Record Subsequent Invoice',
  'Release Purchase Order',
  'Remove Payment Block',
  'SRM: Awaiting Approval',
  'SRM: Change was Transmitted',
  'SRM: Complete',
  'SRM: Created',
  'SRM: Document Completed',
  'SRM: Held',
  'SRM: In Transfer to Execution Syst.',
  'SRM: Incomplete',
  'SRM: Ordered',
  'Vendor creates debit memo',
  'Vendor creates invoice'},
 {'Record Invoice Receipt

### --- Experiments Generation ---

In [15]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic19-cf_seed777_experiments_ga_ablated_output.txt", console=False)

In [16]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [17]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=0.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_Ablated_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/500 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,4508042805_00010,6,1,2,0.317275,0.274549,0.360,0.389286,0.400000,...,0.121429,0.400000,0.050000,0.1,0.000000,0.071429,0.0,0.886350,0.0,1.0
1,0,4508044035_00320,8,1,2,0.424165,0.503330,0.345,0.446429,0.431579,...,0.121429,0.315789,0.050000,0.1,0.000000,0.071429,0.0,0.000000,0.0,0.0
2,0,4507029240_00300,10,1,4,0.346155,0.397309,0.295,0.421429,0.482609,...,0.265286,0.608696,0.051000,0.1,0.002001,0.214286,0.0,0.933362,0.0,1.0
3,0,4507017655_00360,12,1,6,0.397443,0.429886,0.365,0.414286,0.462963,...,0.121429,0.444444,0.050000,0.1,0.000000,0.071429,0.0,0.925347,0.0,1.0
4,0,4507037320_00020,14,1,6,0.320959,0.356919,0.285,0.371429,0.512903,...,0.000000,0.516129,0.000000,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
462,49,4507026467_00001,40,1,0,0.293822,0.292644,0.295,0.496875,0.107229,...,0.079901,0.000000,0.017401,0.0,0.034802,0.062500,0.0,0.000000,0.0,0.0
463,49,4507032796_00001,42,1,0,0.347868,0.255736,0.440,0.489062,0.163218,...,0.050026,0.000000,0.018776,0.0,0.037552,0.031250,0.0,0.000000,0.0,0.0
464,49,4507024372_00001,44,1,0,0.363207,0.281414,0.445,0.526562,0.275824,...,0.139476,0.725275,0.045726,0.0,0.091452,0.093750,0.0,0.000000,0.0,0.0
465,49,4507026083_00001,48,1,0,0.313880,0.282760,0.345,0.485937,0.093939,...,0.032847,0.000000,0.001597,0.0,0.003194,0.031250,0.0,0.000000,0.0,0.0


### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()